# 02 – Candidate ligand–receptor pairs, CD8αα(I) → DC1/aDC1, fetal and postnatal

**Question.** Which S5 pairs are candidates for the two runs, and do the claimed pairs (XCL1 → XCR1, XCL2 → XCR1)
pass the expression filter in fetal and in postnatal cells?

Same method as thymus case 2 (notebook 02): orientation from the CellPhoneDB v2 flags, then a 10% detection filter on
both sides. The only change: each population is split by age class, and a pair is kept if it passes in **at least one
age class**. Both runs then use **the same list of L–R links** (notebook 03), so that their networks can be compared.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
DB = THY / "cellphonedb"
OUT = ROOT / "results/thymus_case3_xcl1_age/02_ligand_receptor_candidates"
OUT.mkdir(parents=True, exist_ok=True)

AGES = ["fetal", "postnatal"]
SENDERS = [f"CD8αα(I) | {a}" for a in AGES]
RECEIVERS = [f"{p} | {a}" for p in ["DC1", "aDC1"] for a in AGES]

s5 = pd.read_csv(THY / "aay3224_table-s5.csv", header=1, low_memory=False).drop(columns="Unnamed: 0")
prot = pd.read_csv(DB / "protein_input.csv").set_index("uniprot")
cplx = pd.read_csv(DB / "complex_input.csv").set_index("complex_name")
gene = pd.read_csv(DB / "gene_input.csv").drop_duplicates("uniprot").set_index("uniprot")
# The count matrix uses older names for the nectins (PVRL1-3); rename them to match
gene["gene_name"] = gene.gene_name.replace({"NECTIN1": "PVRL1", "NECTIN2": "PVRL2", "NECTIN3": "PVRL3"})
print("S5 pairs:", len(s5))

S5 pairs: 710


## 1. Partners → genes, receptor flags and orientation

Same rules as thymus case 1 (notebook 03): genes are matched by symbol through the UniProt ID; manual ligands first,
then the receptor flag, then the integrin flag; pairs that remain ambiguous are kept in both orientations.

In [2]:
def partner_info(p):
    kind, name = p.split(":", 1)
    if kind == "simple":
        if name not in prot.index:
            return dict(genes=None, receptor=None, integrin=None)
        return dict(genes=[gene.gene_name.get(name)], receptor=bool(prot.receptor[name]),
                    integrin=bool(prot.integrin[name]))
    row = cplx.loc[name]
    units = [u for u in row[["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4"]].dropna()]
    return dict(genes=[gene.gene_name.get(u) for u in units], receptor=bool(row.receptor), integrin=bool(row.integrin))

rows = []
for _, r in s5.iterrows():
    a, b = partner_info(r.partner_a), partner_info(r.partner_b)
    rows.append(dict(pair=r.interacting_pair, partner_a=r.partner_a, partner_b=r.partner_b,
                     genes_a=a["genes"], genes_b=b["genes"], receptor_a=a["receptor"], receptor_b=b["receptor"],
                     integrin_a=a["integrin"], integrin_b=b["integrin"]))
lr = pd.DataFrame(rows)
print("pairs with a partner missing from the database:",
      lr.loc[lr.genes_a.isna() | lr.genes_b.isna(), "pair"].tolist())
lr = lr.dropna(subset=["genes_a", "genes_b"]).reset_index(drop=True)

MANUAL_LIGANDS = {"TNFSF11", "CXCL16", "HBEGF", "NRG1", "SEMA4C", "SEMA4D"}
def orient(r):
    sa = set(r.genes_a) if r.partner_a.startswith("simple") else set()
    sb = set(r.genes_b) if r.partner_b.startswith("simple") else set()
    if sa & MANUAL_LIGANDS and not sb & MANUAL_LIGANDS: return "a", "manual"
    if sb & MANUAL_LIGANDS and not sa & MANUAL_LIGANDS: return "b", "manual"
    if r.receptor_a != r.receptor_b:                    return ("b" if r.receptor_a else "a"), "receptor flag"
    if r.integrin_a != r.integrin_b:                    return ("b" if r.integrin_a else "a"), "integrin"
    return None, "ambiguous"

lr[["ligand_side", "orientation_rule"]] = lr.apply(lambda r: pd.Series(orient(r)), axis=1)

def as_oriented(r, side):
    lig, rec = ("a", "b") if side == "a" else ("b", "a")
    return dict(pair=r.pair, ligand=r["partner_" + lig].split(":", 1)[1], receptor=r["partner_" + rec].split(":", 1)[1],
                ligand_genes=r["genes_" + lig], receptor_genes=r["genes_" + rec],
                ligand_is_partner_a=(lig == "a"), orientation_rule=r.orientation_rule)

oriented = pd.DataFrame([as_oriented(r, s) for _, r in lr.iterrows()
                         for s in ([r.ligand_side] if r.ligand_side else ["a", "b"])])
oriented["ligand"] = [g[0] if l in gene.index else l for l, g in zip(oriented.ligand, oriented.ligand_genes)]
oriented["receptor"] = [g[0] if l in gene.index else l for l, g in zip(oriented.receptor, oriented.receptor_genes)]
print(lr.orientation_rule.value_counts().to_string())
oriented[oriented.pair.isin(["XCL1_XCR1", "XCL2_XCR1", "CD74_MIF", "CCL19_CCR7"])][
    ["pair", "ligand", "receptor", "ligand_is_partner_a", "orientation_rule"]]

pairs with a partner missing from the database: ['RIF1_NOTCH1']
orientation_rule
receptor flag    431
integrin         208
ambiguous         55
manual            15


,pair,ligand,receptor,ligand_is_partner_a,orientation_rule
19,CCL19_CCR7,CCL19,CCR7,True,receptor flag
190,CD74_MIF,MIF,CD74,False,receptor flag
384,XCL1_XCR1,XCL1,XCR1,True,receptor flag
385,XCL2_XCR1,XCL2,XCR1,True,receptor flag


## 2. Expression per population and age class

Fraction of cells with at least one raw count, for every S5 gene, in each population × age class (cells of
notebook 01).

In [3]:
cells = pd.read_csv(ROOT / "results/thymus_case3_xcl1_age/01_cell_populations/selected_cells.csv", index_col=0)
raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")

all_genes = sorted({g for gs in pd.concat([oriented.ligand_genes, oriented.receptor_genes]) for g in gs if g})
gene_pos = pd.Series(np.arange(raw.n_vars), index=raw.var_names)
all_genes = [g for g in all_genes if g in gene_pos.index]
assert not set(all_genes) & set(raw.var_names[raw.var_names.duplicated()])

row_pos = np.sort(raw.obs_names.get_indexer(cells.index))
X = raw.X[row_pos][:, gene_pos[all_genes].values]
c = cells.loc[raw.obs_names[row_pos]]
pop_of_row = (c.population + " | " + c.age_class).values

frac = pd.DataFrame((X > 0).toarray(), columns=all_genes).groupby(pop_of_row).mean()
frac.shape

/tmp/ipykernel_447/2396623013.py:2: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


(6, 516)

In [4]:
THRESHOLD = 0.10

def side_fraction(genes, pops):
    genes = [g for g in genes if g in frac.columns]
    if not genes:
        return pd.Series(0.0, index=pops)
    return frac.loc[pops, genes].min(axis=1)               # complex: weakest subunit

def passes(r, age):
    s = [p for p in SENDERS if p.endswith(age)]
    t = [p for p in RECEIVERS if p.endswith(age)]
    return side_fraction(r.ligand_genes, s).max() >= THRESHOLD and side_fraction(r.receptor_genes, t).max() >= THRESHOLD

out = []
for _, r in oriented.iterrows():
    ok = {a: passes(r, a) for a in AGES}
    if any(ok.values()):
        fl, fr = side_fraction(r.ligand_genes, SENDERS), side_fraction(r.receptor_genes, RECEIVERS)
        out.append(dict(pair=r.pair, ligand=r.ligand, receptor=r.receptor, orientation_rule=r.orientation_rule,
                        ligand_is_partner_a=r.ligand_is_partner_a, passes_fetal=ok["fetal"],
                        passes_postnatal=ok["postnatal"],
                        ligand_frac_fetal=fl[SENDERS[0]], ligand_frac_postnatal=fl[SENDERS[1]],
                        max_receptor_frac=fr.max(), receptor_pops=", ".join(fr.index[fr >= THRESHOLD])))
cand = pd.DataFrame(out)
print("candidates:", len(cand))
pd.crosstab(cand.passes_fetal, cand.passes_postnatal)

candidates: 47


passes_postnatal,False,True
passes_fetal,,
False,0,2
True,18,27


## 3. Comparison with the S5 values

As in case 2: the S5 column is read in the ligand → receptor orientation; CD8αα(I) is `CD8aa(I)` and aDC1 uses the
merged `aDC` column. S5 has no age split, so its value is the same for both runs.

In [5]:
TO_S5 = {"CD8αα(I)": "CD8aa(I)", "aDC1": "aDC"}
s5v = s5.set_index("interacting_pair")

def s5_best(r):
    best = (0.0, "")
    for t in ["DC1", "aDC1"]:
        S, T = TO_S5["CD8αα(I)"], TO_S5.get(t, t)
        v = float(s5v.loc[r.pair, f"{S}_{T}" if r.ligand_is_partner_a else f"{T}_{S}"])
        if v > best[0]:
            best = (v, f"CD8αα(I) -> {t}")
    return pd.Series(best, index=["s5_max_mean", "s5_best_pair"])

cand[["s5_max_mean", "s5_best_pair"]] = cand.apply(s5_best, axis=1)
show = ["pair", "ligand", "receptor", "passes_fetal", "passes_postnatal", "ligand_frac_fetal", "ligand_frac_postnatal",
        "receptor_pops", "s5_max_mean"]
cand.sort_values("s5_max_mean", ascending=False)[show].head(35).round(2)

,pair,ligand,receptor,passes_fetal,passes_postnatal,ligand_frac_fetal,ligand_frac_postnatal,receptor_pops,s5_max_mean
17,CD74_MIF,MIF,CD74,True,True,0.87,0.62,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",3.86
3,APP_CD74,APP,CD74,True,True,0.29,0.18,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",2.90
12,MIF_TNFRSF14,MIF,TNFRSF14,True,True,0.87,0.62,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",1.42
36,XCL1_XCR1,XCL1,XCR1,True,True,0.29,0.12,"DC1 | fetal, DC1 | postnatal",1.12
1,LGALS9_CD44,LGALS9,CD44,True,True,0.15,0.11,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",1.10
30,CD28_CD86,CD28,CD86,True,True,0.71,0.43,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",0.84
37,XCL2_XCR1,XCL2,XCR1,True,False,0.16,0.05,"DC1 | fetal, DC1 | postnatal",0.81
5,CD40LG_CD40,CD40LG,CD40,True,False,0.20,0.06,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",0.81
20,CD80_CD28,CD28,CD80,True,True,0.71,0.43,"aDC1 | fetal, aDC1 | postnatal",0.75
43,CD8 receptor_LCK,LCK,CD8 receptor,True,True,0.82,0.79,"DC1 | fetal, DC1 | postnatal, aDC1 | fetal, aD...",0.74


**Reading.**
- 47 oriented pairs pass the filter in at least one age class: 27 in both, 18 in fetal cells only and 2 in postnatal
  cells only.
- **Most fetal-only pairs have a ligand whose share of expressing CD8αα(I) cells falls below 10% after birth:**
  XCL2, CD40LG, CRTAM, LTA, CD226 and TNFSF14. The receptors on DC1 / aDC1 change much less.
- As in case 2, the ranking by S5 mean starts with the ubiquitous MIF / APP → CD74. XCL1 → XCR1 comes fourth.

## 4. The claimed pairs, by age

In [6]:
claim_genes = ["XCL1", "XCL2", "XCR1", "CCR7", "CD40LG", "CD40", "CD28", "CD80", "CD86", "PDCD1", "CD274", "CRTAM", "CADM1"]
frac.loc[SENDERS + RECEIVERS, claim_genes].round(2)

,XCL1,XCL2,XCR1,CCR7,CD40LG,CD40,CD28,CD80,CD86,PDCD1,CD274,CRTAM,CADM1
CD8αα(I) | fetal,0.29,0.16,0.00,0.18,0.20,0.00,0.71,0.00,0.00,0.46,0.00,0.15,0.07
CD8αα(I) | postnatal,0.12,0.05,0.00,0.10,0.06,0.00,0.43,0.00,0.00,0.29,0.00,0.05,0.00
DC1 | fetal,0.01,0.00,0.58,0.04,0.01,0.41,0.02,0.10,0.52,0.11,0.01,0.10,0.79
DC1 | postnatal,0.00,0.00,0.49,0.06,0.01,0.47,0.03,0.07,0.51,0.08,0.01,0.14,0.72
aDC1 | fetal,0.06,0.01,0.02,0.90,0.04,0.91,0.05,0.75,0.63,0.13,0.65,0.12,0.46
aDC1 | postnatal,0.00,0.00,0.03,0.89,0.00,0.89,0.03,0.54,0.63,0.00,0.37,0.03,0.17


In [7]:
cand[cand.pair.isin(["XCL1_XCR1", "XCL2_XCR1"])][show].round(2)

,pair,ligand,receptor,passes_fetal,passes_postnatal,ligand_frac_fetal,ligand_frac_postnatal,receptor_pops,s5_max_mean
36,XCL1_XCR1,XCL1,XCR1,True,True,0.29,0.12,"DC1 | fetal, DC1 | postnatal",1.12
37,XCL2_XCR1,XCL2,XCR1,True,False,0.16,0.05,"DC1 | fetal, DC1 | postnatal",0.81


**Reading.**
- **XCL1 → XCR1 passes in both age classes:** XCL1 is in 29% of fetal and 12% of postnatal CD8αα(I) cells, and XCR1
  in about half of DC1 cells.
- **XCL2 → XCR1 passes only in fetal cells** (XCL2 in 16% against 5%). As a claimed pair it is kept in both runs
  anyway, so that the two networks have the same L–R links.
- **On the receiver side, the activation genes barely change with age** (CD40 about 90% of aDC1 in both age
  classes). CD80 and CD274 (PD-L1) are lower in postnatal aDC1 (54% against 75%, 37% against 65%), but these rest on
  35 cells only.
- **Several sender ligands of the activation axes are lower after birth:** CD28 (71% → 43%), PDCD1 (46% → 29%),
  CD40LG (20% → 6%) and CRTAM (15% → 5%).

In [8]:
cand.to_csv(OUT / "candidates_cd8aa_to_dc1_by_age.csv", index=False)
oriented.drop(columns=["ligand_genes", "receptor_genes"]).to_csv(OUT / "s5_oriented.csv", index=False)
frac.T.round(3).to_csv(OUT / "expression_fraction_by_population_and_age.csv")
print("written to", OUT)

written to ../../results/thymus_case3_xcl1_age/02_ligand_receptor_candidates


## Summary

| | |
|---|---|
| Candidates | 47 oriented pairs pass in at least one age class (27 in both, 18 fetal only, 2 postnatal only) |
| XCL1 → XCR1 | passes in both age classes (ligand 29% fetal, 12% postnatal) |
| XCL2 → XCR1 | passes in fetal cells only (16% against 5%); kept in both runs as a claimed pair |

Next (notebook 03): the cells and the shared list of L–R links for the two runs.